# Train the fretlift transcription model on Colab

1. **Runtime -> Change runtime type -> GPU** (a T4 is fine).
2. Run the cells in order. Datasets download to the Colab machine's own disk,
   which is fast and doesn't use your Drive space. Runs (checkpoints, history)
   are saved to Google Drive.

**If the session disconnects:** run the Setup cells and the data cell again,
then re-run the training cell that was running. It picks up from its last
finished epoch.

See `docs/TRAINING.md` in the repo for what each step does.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Setup

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/fretlift"   # runs and reports: small, and they survive disconnects
DATA = "/content/data"                     # datasets: on this machine, downloaded again each session
!mkdir -p $WORK
COMMON = f"--manifest {DATA}/guitarset/manifest.jsonl --cache-dir /content/cache"
SPLIT = "--val-groups guitarset-player-04 --test-groups guitarset-player-05"

def resume(run):  # continue a run from its last finished epoch, if it has one
    last = f"{WORK}/runs/{run}/last.pt"
    return f"--out {WORK}/runs/{run}" + (f" --resume {last}" if os.path.exists(last) else "")

In [ ]:
REPO = "https://github.com/jdinkha/fretlift.git"
!git clone -q $REPO /content/fretlift || (cd /content/fretlift && git pull -q)
%cd /content/fretlift
# Colab already has a CUDA build of PyTorch, so this only adds what's missing.
!pip install -q -e ".[train,dev]"
!pytest -q

## Step 1: data and the Basic Pitch baseline

GuitarSet is about 1.5 GB and Guitar-TECHS about 4.1 GB; each takes a minute
or two to download here. The baseline needs Basic Pitch; if installing it
conflicts with Colab's packages, skip the baseline cell here and run it in a
local virtual environment instead.

In [ ]:
!fretlift-data guitarset --root {DATA}/guitarset --download
!fretlift-data guitar-techs --root {DATA}/guitar-techs --download
!fretlift-data stats --manifest {DATA}/guitarset/manifest.jsonl --manifest {DATA}/guitar-techs/manifest.jsonl

In [ ]:
!pip install -q basic-pitch
!fretlift-eval --basic-pitch --split test --test-groups guitarset-player-05 {COMMON} --out {WORK}/reports/bp-clean.json
!fretlift-eval --basic-pitch --split test --test-groups guitarset-player-05 {COMMON} --distort high_gain --out {WORK}/reports/bp-high_gain.json
!fretlift-sweep --basic-pitch --split val --val-groups guitarset-player-04 {COMMON} --out {WORK}/reports/bp-sweep.json

## Step 3: train on clean audio

In [ ]:
!fretlift-train {COMMON} {SPLIT} --workers 2 {resume("gs-clean")}

## Step 4: train with augmentation

Optional: upload cabinet impulse responses (WAV) to `$WORK/irs` and add
`--ir-dir {WORK}/irs`. Colab has 2 CPU cores, so augmented epochs are slower
(the CPU makes the audio, the GPU waits).

In [ ]:
!fretlift-train {COMMON} {SPLIT} --workers 2 --augment full --val-distort high_gain {resume("gs-aug")}

## Step 5: add Guitar-TECHS (electric), starting from gs-aug

Starts from the gs-aug weights and writes a new run, so `runs/gs-aug` stays as
it is. Guitar-TECHS player 3 (the only real pieces of music) is held out for
validation, so the best checkpoint is picked on electric guitar too.

In [ ]:
TECHS = f"--manifest {DATA}/guitar-techs/manifest.jsonl"
VAL = "--val-groups guitarset-player-04,guitar-techs-P3 --test-groups guitarset-player-05"
!fretlift-train {COMMON} {TECHS} {VAL} --workers 2 --augment full --val-distort high_gain --init {WORK}/runs/gs-aug/best.pt {resume("gs-techs")}

## Compare

In [ ]:
import json, pathlib
for run in ["gs-clean", "gs-aug", "gs-techs"]:
    p = pathlib.Path(WORK) / "runs" / run / "test_report.json"
    if p.exists():
        r = json.loads(p.read_text())
        for dist in [k for k in r if k != "decode"]:
            s = r[dist]["all"]
            print(f"{run:10s} {dist:10s} note F1 {s['note_f1']:.3f}  tab F1 {s['tab_f1'] or 0:.3f}")
for name in ["bp-clean", "bp-high_gain"]:
    p = pathlib.Path(WORK) / "reports" / f"{name}.json"
    if p.exists():
        s = json.loads(p.read_text())["summary"]["all"]
        print(f"{name:21s} note F1 {s['note_f1']:.3f}  tab F1 {s['tab_f1'] or 0:.3f}")

In [ ]:
import pandas as pd
pd.read_csv(f"{WORK}/runs/gs-techs/history.csv").plot(x="epoch", y=["clean_note_f1", "high_gain_note_f1", "clean_tab_f1"], grid=True)